<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---Prompt-Engineering/blob/dev/Prompt_Engineering_Part_1_(Prompt_Chaining).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Prompt Chaining (tools used: ClaudeAI, ChatGPT-4o, apillm7)

#Prompt 1: Generate a python code for a customer support AI, aimed to be treated for a simple E-Commerce company

#Prompt 2: Awesome, now it works. Let's try to make it so that it stimulates customer flow, in which we try to build simple prompts for the user to identify and choose.

#Prompt 2a: Q: How should customers pick their issue?
# A: Numbered menu (type 1, 2, 3...)

#Prompt 2b: Q: What support categories should be included? (Select all that apply)
# A: Orders & Shipping, Returns & Refunds, Billing & Payments, Technical Support, Account Help, General Inquiry

#Prompt 2c: Q: What happens after a category is chosen?
# A: Guided form-style questions

#Prompt 3: Good progress! We are getting much closer to creating a streamlined customer service flow.
# Now that the user has chosen their prompts, end it with an optional response to create another support ticket (or not) & close the support system after.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import json
from datetime import datetime
from openai import OpenAI

# ─── Free AI Client (No API Key Required) ────────────────────────────────────

client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"  # No real key needed
)

MODEL = "gpt-4o-mini-2024-07-18"

# ─── Company Configuration ────────────────────────────────────────────────────

COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday-Friday, 9 AM - 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}

# ─── Category Menus & Guided Forms ────────────────────────────────────────────

CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number",  "What is your order number? (e.g. ORD-12345)"),
        ("issue",         "What's the issue?\n   1. I haven't received my order\n   2. My order arrived damaged\n   3. I received the wrong item\n   4. I need to change my delivery address\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number?"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n   2. Wrong item received\n   3. Changed my mind\n   4. Item not as described\n   Enter 1-4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",         "What's your billing issue?\n   1. I was charged incorrectly\n   2. My payment was declined\n   3. I need a copy of my invoice\n   4. I want to update my payment method\n   Enter 1-4: "),
        ("order_number",  "Related order number? (press Enter to skip): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",      "Where are you experiencing the issue?\n   1. Website\n   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1-4: "),
        ("description",   "Briefly describe the problem you're experiencing: "),
        ("contact_email", "What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",         "What do you need help with?\n   1. I can't log in\n   2. I want to update my details\n   3. I want to delete my account\n   4. I didn't receive a verification email\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",         "What would you like to know about? (briefly describe): "),
        ("contact_email", "What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}

SYSTEM_PROMPT = (
    "You are a friendly and professional customer support agent for "
    + COMPANY_CONFIG["name"] + ", an " + COMPANY_CONFIG["industry"] + " company.\n\n"
    "You will receive a structured summary of a customer's issue gathered from a guided form.\n"
    "Your job is to:\n"
    "- Greet the customer warmly and acknowledge their specific issue\n"
    "- Provide a clear, empathetic, and helpful resolution or next step\n"
    "- Reference their order number or details where relevant\n"
    "- If the issue requires human escalation, mention they'll be contacted at their email within 1 business day\n"
    "- Keep your response concise (3-5 sentences max)\n\n"
    "Company details:\n"
    "- Support email: " + COMPANY_CONFIG["support_email"] + "\n"
    "- Support hours: " + COMPANY_CONFIG["support_hours"] + "\n"
    "- Return policy: " + COMPANY_CONFIG["return_policy"]
)

# ─── Conversation Logger ──────────────────────────────────────────────────────

class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print("\n Session saved to " + self.log_file + " (ID: " + self.session_id + ")")
        except Exception as e:
            print("\n Could not save log: " + str(e))

# ─── Helpers ─────────────────────────────────────────────────────────────────

def print_divider():
    print("-" * 58)

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print("    " + key + ". " + label)
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print("\n  Let's gather some details about your " + category + " issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = ["Customer category: " + category]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)

def ask_ai(messages):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=512,
        messages=messages,
    )
    return response.choices[0].message.content

# ─── Main Support Flow ────────────────────────────────────────────────────────

def run_support_session(logger):
    show_main_menu()

    while True:
        choice = input("  Enter number (1-6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print("\n  You selected: " + category)
            break
        print("  Please enter a number between 1 and 6.")

    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Connecting you with our support agent...\n")
    print_divider()

    history = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": summary},
    ]

    agent_reply = ask_ai(history)
    history.append({"role": "assistant", "content": agent_reply})
    logger.log("assistant", agent_reply)

    print("Support Agent:\n\n  " + agent_reply + "\n")
    print_divider()

    print("  Need anything else? Type a follow-up question or 'done' to exit.\n")

    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue
        if follow_up.lower() in ("done", "quit", "exit", "no", "nope"):
            history.append({"role": "user", "content": "That's all, thank you!"})
            closing = ask_ai(history)
            print("\nAgent: " + closing + "\n")
            break
        history.append({"role": "user", "content": follow_up})
        logger.log("user", follow_up)
        reply = ask_ai(history)
        history.append({"role": "assistant", "content": reply})
        logger.log("assistant", reply)
        print("\nAgent: " + reply + "\n")

# ─── Entry Point ─────────────────────────────────────────────────────────────

def main():
    print("=" * 58)
    print("  " + COMPANY_CONFIG["name"] + " - Customer Support")
    print("  " + COMPANY_CONFIG["support_hours"])
    print("  " + COMPANY_CONFIG["support_email"])
    print("=" * 58)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except Exception as e:
            print("\n Something went wrong: " + str(e))
            print("  The session will now close safely.")
        finally:
            logger.save()

        again = input("\n  Start a new support session? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print("\n  Thank you for contacting " + COMPANY_CONFIG["name"] + " support. Have a great day!\n")
            break

main()

  Acme Corp - Customer Support
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
----------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
----------------------------------------------------------
  Enter number (1-6): 5

  You selected: Account Help

  Let's gather some details about your Account Help issue.

  What do you need help with?
   1. I can't log in
   2. I want to update my details
   3. I want to delete my account
   4. I didn't receive a verification email
   Enter 1-4:  1
  What email is on your account? james.h.phan@sjsu.edu

  Connecting you with our support agent...

----------------------------------------------------------
Support Agent:

  Hello James,

I'm really sorry to hear you're having trouble logging into your Acme Corp account. Let's get you back in! Please try resetting 